# Tables done right: when a table beats a chart

_Data Wrangling_

**For exact values, lookups, and small mixed-unit summaries, a styled table reads better than any plot.**

A chart is for seeing a shape &mdash; a trend, a cluster, an outlier. A table is for
       reading a value &mdash; the exact number in a known cell. They answer different questions, and
       reaching for a chart when the reader actually wants a number (or the reverse) makes the work harder to
       use, not easier.

       Reach for a table when: exact values matter (a price, a count, a p-value), the reader will
       look up one row, the summary is small and multi-dimensional (a few groups by a few
       metrics), or the columns have mixed units (dollars next to percentages next to days) that no
       single axis can carry.

---

This notebook is a practice scaffold for the **Tables done right: when a table beats a chart** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine

# --- A real bundled dataset: wine chemistry, 178 rows, 13 features + class ---
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["wine_class"] = df["target"].map({0: "class_0", 1: "class_1", 2: "class_2"})

# === Step 1: a SMALL summary (groupby aggregation) — not a raw dump ===
feats = ["alcohol", "flavanoids", "color_intensity", "proline", "hue"]
summary = (
    df.groupby("wine_class")[feats]
      .mean()
      .sort_values("proline", ascending=False)   # sort meaningfully (high proline first)
      .reset_index()
)
# add a clearly-set-off summary row (the overall mean)
overall = df[feats].mean()
overall["wine_class"] = "mean (all)"
summary = pd.concat([summary, overall.to_frame().T[summary.columns]], ignore_index=True)

# === Step 2: STYLE it — decimals/units, color scale, data bars, max highlight ===
styled = (
    summary.style
      .format({                                   # consistent decimals + units per column
          "alcohol": "{:.2f}",
          "flavanoids": "{:.2f}",
          "color_intensity": "{:.2f}",
          "proline": "{:.0f}",                     # mg/L — whole numbers
          "hue": "{:.2f}",
      })
      # heatmap shading (a color scale) on the chemistry columns
      .background_gradient(cmap="Greens", subset=["flavanoids", "color_intensity"])
      # in-cell data bars: the column doubles as a tiny bar chart
      .bar(subset=["proline"], color="#9ecae1")
      # flag the per-column winner so the eye lands on it
      .highlight_max(subset=feats, color="#fff3b0")
      .set_caption("Mean wine chemistry by class — alcohol (%), proline (mg/L)")
      .hide(axis="index")                         # drop the noisy default index
)

# In a notebook this renders as a styled HTML table; export for a report with:
#   styled.to_html("wine_summary.html")
#   # or, for publication-quality output:
#   #   from great_tables import GT
#   #   GT(summary).fmt_number(columns=feats, decimals=2)
styled

## Visualize the data & results

_Mean of five chemistry features for each of the three wine classes (load_wine). Read as a heatmap, this conditionally-formatted table shows which class is high on which feature — bright cells are the big numbers._

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["wine_class"] = df["target"].map({0: "class_0", 1: "class_1", 2: "class_2"})

feats = ["alcohol", "flavanoids", "color_intensity", "hue", "proline"]

# A small group-by-metric summary table: rows = wine classes, cols = features.
summary = df.groupby("wine_class")[feats].mean().round(2)
print(summary.to_string())
#            alcohol  flavanoids  color_intensity   hue  proline
# class_0      13.74        2.98             5.53  1.06  1115.71
# class_1      12.28        2.08             3.09  1.06   519.51
# class_2      13.15        0.78             7.40  0.68   629.90

print(df["wine_class"].value_counts().sort_index())   # 59 / 71 / 48

# For the heatmap (one shared color scale) divide proline by 100 so it
# shares a magnitude range with the other columns:
mat = summary.copy()
mat["proline"] = (mat["proline"] / 100).round(2)
print(mat.to_string())

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate pastes a 4,000-row dataframe of daily sales into the report "so the numbers are all there." Is a table the right call, and how would you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask what the reader actually needs from those 4,000 rows. — _Almost nobody reads 4,000 rows; the real question is usually a trend or a per-group total, not every cell._
- If the question is "how do sales trend over time," replace the table with a line chart. — _A trend is a shape &mdash; a chart shows it instantly where a giant table hides it._
- If exact per-group numbers are needed, aggregate to a small styled summary (e.g. sales by region or month) and show that. — _A small sorted, formatted summary table is readable and gives the exact values; the raw dump gives neither._

**Answer:** A 4,000-row raw dump is the classic table pitfall &mdash; it's neither readable nor useful. Decide by the question: if the reader wants the trend, that's a chart (line over time); if they want exact per-group values, aggregate with a groupby to a small summary table (sales by region/month), then sort it, fix the decimals, label the units, and color-scale the key column. Either way, the 4,000-row block does not belong in the report.

</details>

**Problem 2.** You have a small report comparing three regions on revenue ($), conversion rate (%), and average delivery time (days). Chart or table? Why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the dimensions: 3 groups by 3 metrics, and the metrics have different units. — _A small group-by-metric grid with mixed units is the textbook case for a table, not a single chart axis._
- Note that exact values (a revenue figure, a percentage) will be quoted. — _Quotable exact numbers favor a table; a chart would force estimation off an axis._
- Design it: right-align numbers, put $, %, and days in the headers, one decimal format per column, sort by revenue, color-scale each metric. — _That makes the mixed-unit comparison legible and lets the eye find the strong/weak region per metric._

**Answer:** A table. It's a tiny multi-dimensional summary (3&times;3) with mixed units ($, %, days) that no single chart axis can carry, and the values are exact and quotable. Design it well: units in the headers, right-aligned numbers, one decimal format per column, sorted by revenue, with a calm color scale (or data bars) per metric so the strongest and weakest region pop without burying the numbers. If you also want the shape, add a small chart beside it &mdash; report tables and charts work together.

</details>

**Problem 3.** Someone styles a summary table by shading every cell in a bright rainbow gradient and bolding nothing. What's wrong, and what's the rule of thumb for in-cell emphasis?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice that if every cell is loudly colored, no cell stands out. — _Emphasis is relative &mdash; coloring everything is the same as coloring nothing, and it strains the eye._
- Notice a rainbow scale has no inherent order, so high-vs-low isn't obvious. — _A sequential (single-hue, light&rarr;dark) scale maps magnitude monotonically; rainbow does not._
- Restyle: one calm sequential color scale per metric, and bold just the key number in each row. — _Sparing, ordered emphasis guides the eye to the answer while the exact digits stay readable._

**Answer:** Over-coloring kills readability: if every cell shouts, nothing stands out, and a rainbow scale has no natural high&rarr;low order so it doesn't even encode magnitude. The rule of thumb is emphasize sparingly &mdash; bold the one key number per row, use a single sequential color scale (light&rarr;dark) per metric so big values read as "more," and leave most cells plain. The shading should guide the eye to the answer, not compete with the numbers it's printed over.

</details>